# Visualization of segment contributions as a function of solvent ratio (BLI_GAT_SEI)
Put a SMILES string to use the BLI-GAT-SEI model.


In [1]:
%load_ext autoreload
%autoreload 2
import numpy as np
import SSC
from SEI_visualization_utils import visualize_ratio_sweep

## BLI-GAT-SEI model

In [2]:
BLI_segment = SSC.src.SSCAnalyzer.SSCAnalyzer_v1p3("BLI_segment_sum_contribution")

founded model: ./Trained_models/BLI_segment_sum_contribution
Device: cuda
#params: 184707
Loading params from ./Trained_models/BLI_segment_sum_contribution/final.pth


/home/ainshin/miniconda3/envs/ksw_pyg_dgl/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


Sculptor Index: (6, 2, 0)


## Put/replace SMILES 

Run by only changing `SMILES`.

In [3]:
# Put/replace SMILES, ratio
SMILES = "C1=CC=C(C=C1)N"          # molecule SMILES
elu1_smiles = "CCCCCC"           # eluent1 SMILES (ex: "CCCCCC")
elu2_smiles = "CCOC(C)=O"        # eluent2 SMILES (ex: "CCOC(C)=O")
ratio1 = [0.5]                   # eluent1 ratio (ex: [1.0, 0.9, ..., 0.0])
ratio2 = [0.5]                   # eluent2 ratio list (ex: [0.0, 0.1, ..., 1.0])

if not (SMILES and elu1_smiles and elu2_smiles and len(ratio1) and len(ratio2)):
    raise ValueError("Put/replace SMILES, elu1_smiles, elu2_smiles, ratio1, ratio2.")
if len(ratio1) != len(ratio2) or not np.allclose(np.add(ratio1, ratio2), 1.0):
    raise ValueError("The sum of ratio1 and ratio2 must be 1..")

custom_table, custom_scores = visualize_ratio_sweep(
    BLI_segment, SMILES,
    hex_smiles=elu1_smiles, ea_smiles=elu2_smiles, ea_fractions=list(ratio2),
    label1=elu1_smiles, label2=elu2_smiles,
)

custom_summary = custom_table.pivot_table(
    index=[elu2_smiles, elu1_smiles, "Rf_pred"], columns=["frag", "frag_smiles"], values="contribution"
)
custom_summary.round(4)

,,frag,0,1
,,frag_smiles,*N,c1ccccc1
CCOC(C)=O,CCCCCC,Rf_pred,,
0.5,0.5,0.653102,-0.0718,0.7449


## HEX/EA

Run by only changing `SMILES`.

In [4]:
SMILES = "C1=CC=C(C=C1)N"

HEX_SMILES = "CCCCCC"
EA_SMILES = "CCOC(C)=O"
EA_FRACTIONS = np.round(np.linspace(0.0, 1.0, 11), 1)  # 0.0, 0.1, ..., 1.0


sweep_table, sweep_scores = visualize_ratio_sweep(
    BLI_segment, SMILES,
    hex_smiles=HEX_SMILES, ea_smiles=EA_SMILES, ea_fractions=EA_FRACTIONS,
    label1="HEX", label2="EA",
)

## DCM / MeOH

Run by only changing `SMILES`.

In [5]:
SMILES = "C1=CC=C(C=C1)N"
DCM_SMILES = "ClCCl"
MEOH_SMILES = "CO"

dcm_table, dcm_scores = visualize_ratio_sweep(
    BLI_segment, SMILES,
    hex_smiles=DCM_SMILES, ea_smiles=MEOH_SMILES,
    label1="DCM", label2="MeOH",
)

dcm_summary = dcm_table.pivot_table(
    index=["MeOH", "DCM", "Rf_pred"], columns=["frag", "frag_smiles"], values="contribution"
)
dcm_summary.round(4)

,,frag,0,1
,,frag_smiles,*N,c1ccccc1
MeOH,DCM,Rf_pred,,
0.0,1.0,0.307684,-0.3427,-0.2317
0.1,0.9,0.732396,0.1913,0.7763
0.2,0.8,0.748705,0.1534,0.8161
0.3,0.7,0.713831,0.1218,0.7730
0.4,0.6,0.684166,0.0753,0.7386
0.5,0.5,0.640181,0.0334,0.6643
0.6,0.4,0.579391,0.0183,0.5040
0.7,0.3,0.524014,-0.0033,0.3262
